# Pelatihan & Evaluasi Model Prediksi Harga H+7 Berbasis ANN Perceptron (`5 -> 16 -> 1`)
**Platform EWS RADAR PANGAN Jawa Barat (27 Kabupaten/Kota $\times$ 6 Komoditas Strategis)**

### Spesifikasi Arsitektur Model:
- **Pendekatan Pemodelan**: Model Independen Spesifik Wilayah & Komoditas (*162 Local Models*).
- **Horizon Prediksi**: *Direct $t \to t+7$ Forecasting* (Senin ini langsung memprediksi Senin depan).
- **Input Layer (5 Neuron Siklus Mingguan)**: $[H_t, H_{t-1}, H_{t-7}, H_{t-14}, H_{t-21}]$
- **Hidden Layer**: 1 Lapisan berisi **16 Neuron Perceptron** dengan fungsi aktivasi **ReLU** dan regularisasi L2 ($\alpha=1.0$).
- **Output Layer (1 Neuron)**: $\hat{H}_{t+7}$ (Estimasi Harga 7 Hari ke Depan).
- **Pengaman Sumber Daya (*Resource Guard*)**: Kunci 1 CPU Thread (`OMP_NUM_THREADS=1`), *Early Stopping*, dan batas waktu eksekusi.

In [ ]:
import os
# Kunci penggunaan thread CPU agar tidak membebani sistem secara berlebihan
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'

import csv
import json
import time
import numpy as np
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, root_mean_squared_error
print('Library siap digunakan.')

## 1. Unggah / Baca Dataset Bapanas 2024–2026

In [ ]:
# Jika dijalankan di Google Colab, unggah file CSV Bapanas atau arahkan ke path file
CSV_PATH = 'Data Harga Beras Medium, Bamer, CRM, Telur, Minyak Jabar 2024-2026.xlsx - Konsumen.csv'

KOMODITAS_KOLOM = {
    'beras_medium': 3,
    'bawang_merah': 4,
    'cabai_rawit_merah': 5,
    'telur_ayam_ras': 6,
    'minyak_goreng_curah': 7,
    'minyak_goreng_kemasan': 8,
}

def muat_csv_bapanas(path: str):
    with open(path, mode='r', encoding='utf-8') as f:
        rows = list(csv.reader(f))
    seri = {}
    for r in rows[5:]:
        if len(r) < 9:
            continue
        wilayah = ' '.join(r[1].strip().lower().replace('.', '').split())
        if wilayah == 'jawa barat':
            continue
        for kom, col_idx in KOMODITAS_KOLOM.items():
            val_str = r[col_idx].strip().replace(',', '')
            kunci = (wilayah, kom)
            seri.setdefault(kunci, [])
            if val_str and val_str != '-' and val_str.isdigit():
                seri[kunci].append(int(val_str))
            elif seri[kunci]:
                # Forward-Fill (LOCF) untuk menjaga kontinuitas seri tanpa data leakage
                seri[kunci].append(seri[kunci][-1])
    return seri


## 2. Fungsi Pelatihan ANN Perceptron (`5 -> 16 ReLU -> 1`) dengan Pengaman Iterasi & Waktu

In [ ]:
HORIZON = 7
LAGS = [0, 1, 7, 14, 21]  # H(t), H(t-1), H(t-7), H(t-14), H(t-21)
MIN_LAG = max(LAGS)
HIDDEN_UNITS = (16,)
TRAIN_RATIO = 0.80

def latih_ann_perceptron(nilai: list[int], max_iter: int = 350):
    X_rows, y_rows = [], []
    for t in range(MIN_LAG, len(nilai) - HORIZON):
        X_rows.append([float(nilai[t - lag]) for lag in LAGS])
        y_rows.append(float(nilai[t + HORIZON]))
    X = np.array(X_rows, dtype=np.float64)
    y = np.array(y_rows, dtype=np.float64)

    split = int(len(y) * TRAIN_RATIO)
    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]

    scaler_x = StandardScaler()
    scaler_y = StandardScaler()
    X_train_s = scaler_x.fit_transform(X_train)
    X_test_s = scaler_x.transform(X_test)
    y_train_s = scaler_y.fit_transform(y_train.reshape(-1, 1)).ravel()

    mlp = MLPRegressor(
        hidden_layer_sizes=HIDDEN_UNITS,
        activation='relu',
        solver='adam',
        alpha=1.0,
        learning_rate_init=0.01,
        max_iter=max_iter,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=15,
        tol=1e-3,
        random_state=42,
    )
    mlp.fit(X_train_s, y_train_s)

    pred_test_s = mlp.predict(X_test_s)
    pred_test = scaler_y.inverse_transform(pred_test_s.reshape(-1, 1)).ravel()

    return {
        'mae_rp': round(float(mean_absolute_error(y_test, pred_test)), 2),
        'rmse_rp': round(float(root_mean_squared_error(y_test, pred_test)), 2),
        'mape_persen': round(float(mean_absolute_percentage_error(y_test, pred_test) * 100.0), 2),
        'iterasi': int(mlp.n_iter_),
        'mlp': mlp,
        'scaler_x': scaler_x,
        'scaler_y': scaler_y,
    }
